# Week 3, Lab 1 — Your first Crew

CrewAI = roles + tasks + a process. Point `LLM` at Ollama or the Colab compat server.


In [ ]:
WEEK = 'Week 3'
LAB = 'Lab 1 — first crew'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


In [ ]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn crewai
else:
    %pip install -q crewai ollama


In [ ]:
cfg = openai_client_kwargs()
from crewai import LLM, Agent, Task, Crew, Process

llm = LLM(
    model=f"openai/{cfg['model']}",
    api_key=cfg["api_key"],
    base_url=cfg["base_url"],
)
print("CrewAI LLM ->", cfg)


In [ ]:
researcher = Agent(
    role="Researcher",
    goal="Find accurate information about the given topic using what you know and keep it factual.",
    backstory="A careful research analyst who prefers short bullet facts.",
    llm=llm,
    verbose=True,
)
writer = Agent(
    role="Writer",
    goal="Turn research notes into a 4-sentence student-friendly explanation.",
    backstory="A teacher who hates jargon.",
    llm=llm,
    verbose=True,
)
t1 = Task(description="List 4 facts about MCP (Model Context Protocol).", expected_output="4 short bullets.", agent=researcher)
t2 = Task(description="Write a 4-sentence explanation of MCP for beginners, using the research.", expected_output="4 sentences.", agent=writer)
crew = Crew(agents=[researcher, writer], tasks=[t1, t2], process=Process.sequential, verbose=True)
print(crew.kickoff())


Keep tasks tiny on small models. **Next:** backstory ablations.
